# Notebook 03 - SageMaker Pipeline (Singlish Marker Keyboard)

**ITI113 Team 16 · MLOps & Deployment (Joseph, s1602)**

This notebook builds the training pipeline on SageMaker. It is adapted from the lecturer's Notebook 03 (the heart-disease preprocessing-bundle version). We keep the lecturer's pipeline structure and conventions, and change the parts that are specific to our text problem.

**What changed from the lecturer's version, and why (student notes):**

- **Problem type.** The lecturer's example is tabular heart-disease data with binary classification. Ours is text: we predict the sentence-final marker (a particle, an emoji, or none) from a message. This is multi-class classification.
- **Preprocessing.** The lecturer engineers tabular features and fits a `StandardScaler`. We instead read the SMS corpus, apply our Option C labelling through the shared `singlish_labelling` module, remove duplicates, and scrub PII. There is no scaler to save.
- **One Pipeline artefact.** The lecturer saves the model and the scaler separately in a bundle. We fold the vectoriser (TF-IDF) and the classifier into a single `sklearn.Pipeline` and save that one object. This is the same one-artefact approach we used in the ITI112 assignment, and it avoids a separate preprocessing hand-off at the endpoint.
- **Metrics.** The lecturer reports AUC-ROC (a binary metric). We report **macro-F1** and **top-3 accuracy**, which suit our imbalanced multi-class problem.
- **Quality gate.** The lecturer's pipeline gates model registration on test AUC-ROC. We gate on **macro-F1** instead, since AUC-ROC does not apply here.
- **Two models.** The classifier inside the training script is chosen by a hyperparameter, so the same pipeline can train and serve **both** our models (Linear SVM and a tree ensemble). This meets the requirement that the MLOps flow supports more than one model.

**Assumptions (student notes):**
- The corpus is already in S3 (Notebook 01B). The shared `singlish_labelling.py` is available to copy into the pipeline source folder.
- We keep Studio on a small instance and run the heavy work as SageMaker jobs, per the lecturer's guidance.
- There is no visual pipeline editor in this course setup; we inspect runs through the SageMaker console and CloudWatch (and the lecturer's troubleshooting notebook).

**Run order:** 01A (MLflow app) → 01B (S3) → 01/02 (EDA + baseline) → **03 (this)** → 04 (Gradio demo).

## 0. Install packages

Same as the lecturer's setup. If packages upgrade, restart the kernel before continuing.

In [1]:
# Lecturer's known-good install (pinned to the 2.x line for this Studio environment).
# After running, restart the kernel before continuing.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sagemaker
print(sagemaker.__version__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


2.257.6


## 1. AWS / SageMaker configuration

Our Team 16 values. This mirrors the lecturer's config cell with our identifiers.

In [3]:
import boto3, sagemaker, os
from sagemaker.workflow.pipeline_context import PipelineSession

# ---- Team 16 config ----
TEAM_ID    = "team16"
STUDENT_ID = "s1602"
SEMESTER   = "26S1"
REGION     = "ap-southeast-1"
PROJECT    = "singlish-keyboard"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT}"

# Where the labelled corpus lives / will be read from (Notebook 01B staged the raw corpus)
RAW_CORPUS_KEY = f"{PREFIX}/raw/smsCorpus_en_2015.03.09_all.json"

# Instance types - keep Studio light, run jobs on small instances.
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"

PIPELINE_NAME    = f"iti113-{TEAM_ID}-singlish-pipeline"
LOCAL_PIPELINE_SRC = "src"           # scripts live here and ship to the jobs
os.makedirs(LOCAL_PIPELINE_SRC, exist_ok=True)

boto_session = boto3.Session(region_name=REGION)
sm_session   = sagemaker.Session(boto_session=boto_session)
pipeline_session = PipelineSession(boto_session=boto_session)
role = sagemaker.get_execution_role()

print("Team:", TEAM_ID, "| Region:", REGION)
print("Role:", role)
print("Bucket/prefix:", f"s3://{BUCKET}/{PREFIX}")
print("Pipeline name:", PIPELINE_NAME)

Team: team16 | Region: ap-southeast-1
Role: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team16
Bucket/prefix: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard
Pipeline name: iti113-team16-singlish-pipeline


## 2. Put the shared labelling module into the pipeline source folder

The preprocessing job runs in its own container, so it needs its own copy of `singlish_labelling.py`. We copy the shared module into `src/` so it ships with the job. This is what keeps training and serving using the **same** labelling logic (no train/serve skew).

In [4]:
import shutil, os

# The shared module should be in this Studio folder (same one used by the notebooks).
SRC_MODULE = "../src/singlish_labelling.py"
assert os.path.exists(SRC_MODULE), (
    "singlish_labelling.py not found in this folder. "
    "Upload it here first, then re-run."
)
shutil.copy(SRC_MODULE, os.path.join(LOCAL_PIPELINE_SRC, "singlish_labelling.py"))
print("Copied singlish_labelling.py into", LOCAL_PIPELINE_SRC)
print("src/ now contains:", os.listdir(LOCAL_PIPELINE_SRC))

Copied singlish_labelling.py into src
src/ now contains: ['singlish_labelling.py', 'preprocess.py', 'train.py', 'inference.py']


## 3. Preprocessing script

This replaces the lecturer's tabular preprocessing. It reads the raw corpus, applies Option C labelling via the shared module, removes duplicates, scrubs PII, and writes train/test CSVs to the Processing output folder.

Because the vectoriser (TF-IDF) is part of the model Pipeline (next step), there is **no scaler to fit or save here** - this script only produces the labelled `context` (X) and `label` (Y) splits.

In [5]:
%%writefile src/preprocess.py
"""SageMaker Processing Job - Singlish marker data preparation (Option C).

Reads the raw NUS SMS Corpus JSON, applies Option C labelling via the shared
singlish_labelling module, removes duplicates, scrubs PII, and writes
train/test splits. No scaler is fitted here: TF-IDF lives in the model Pipeline.

The 'none' class is capped in the TRAINING set only (after the split); the test
set keeps its natural, real-world distribution. This matches the model-development
notebook so the deployed model equals the reported champion.
"""
import os
import sys
import json
import argparse
import pandas as pd

# The shared module is shipped as a processing input mounted here.
sys.path.insert(0, "/opt/ml/processing/input/module")
import singlish_labelling as SL

parser = argparse.ArgumentParser()
parser.add_argument("--test-size",    type=float, default=0.20)
parser.add_argument("--random-state", type=int,   default=42)
parser.add_argument("--min-class-count", type=int, default=100)
# Cap 'none' in TRAINING ONLY to (cap-multiple x largest marker). <=0 disables.
parser.add_argument("--cap-multiple", type=int, default=3)
args = parser.parse_args()

input_dir  = "/opt/ml/processing/input"
output_dir = "/opt/ml/processing/output"
os.makedirs(output_dir, exist_ok=True)

# --- locate the corpus json in the input folder ---
json_name = None
for f in os.listdir(input_dir):
    if f.endswith(".json"):
        json_name = f
        break
if json_name is None:
    raise FileNotFoundError(f"No .json corpus found in {input_dir}")
raw = json.loads(open(os.path.join(input_dir, json_name), encoding="utf-8").read())

# --- extract message text (same structure as Notebook 01) ---
def get_messages(raw):
    node = raw
    for key in ("smsCorpus", "message"):
        if isinstance(node, dict) and key in node:
            node = node[key]
    return node if isinstance(node, list) else [node]

def extract_text(m):
    if isinstance(m, dict):
        t = m.get("text")
        if isinstance(t, dict) and "$" in t:
            return str(t["$"])
        if isinstance(t, str):
            return t
    return str(m) if isinstance(m, str) else ""

df = pd.DataFrame({"text": [extract_text(m) for m in get_messages(raw)]})
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"Loaded {len(df):,} messages")

# --- scrub PII, then Option C label (dedup handled inside build_labelled_frame) ---
df["scrubbed"] = df["text"].apply(SL.scrub_pii)
labelled = SL.build_labelled_frame(
    df.rename(columns={"scrubbed": "text_for_labelling"}),
    text_col="text_for_labelling",
    use_emoji=True,
    drop_duplicates=True,
)
data = labelled[["context", "label"]].copy()
print("Rows after dedup+label:", len(data))

# --- drop rare PARTICLE classes (emoji classes protected), same rule as the notebook ---
counts = data["label"].value_counts()
drop_particles = [c for c in counts[counts < args.min_class_count].index
                  if c != "none" and not str(c).startswith("emo_")]
if drop_particles:
    print("Dropping rare particle classes:", drop_particles)
    data = data[~data["label"].isin(drop_particles)].reset_index(drop=True)

# --- also drop any class with < 2 examples so stratify works ---
vc = data["label"].value_counts()
too_rare = vc[vc < 2].index.tolist()
if too_rare:
    data = data[~data["label"].isin(too_rare)].reset_index(drop=True)

from sklearn.model_selection import train_test_split
X = data["context"].fillna("")
y = data["label"]
# Split on the NATURAL distribution first (test set stays real-world).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=args.test_size, random_state=args.random_state, stratify=y)

# --- Cap 'none' in the TRAINING set only (test set untouched) ---
# Same logic as the model-development notebook so the deployed model matches the champion.
if args.cap_multiple and args.cap_multiple > 0:
    train_df = pd.DataFrame({"context": X_train, "label": y_train})
    non_none = train_df[train_df["label"] != "none"]["label"].value_counts()
    if len(non_none) > 0:
        largest_marker = int(non_none.max())
        cap = largest_marker * args.cap_multiple
        none_train = train_df[train_df["label"] == "none"]
        if len(none_train) > cap:
            keep_none = none_train.sample(n=cap, random_state=args.random_state)
            train_df = pd.concat(
                [train_df[train_df["label"] != "none"], keep_none]
            ).reset_index(drop=True)
            print(f"Training 'none' capped to {cap:,} "
                  f"(= {args.cap_multiple} x largest marker {largest_marker:,}).")
        X_train, y_train = train_df["context"], train_df["label"]
else:
    print("Capping disabled (cap-multiple <= 0).")

# --- write splits (X is text, Y is label) ---
pd.DataFrame({"context": X_train}).to_csv(os.path.join(output_dir, "train_features.csv"), index=False)
pd.DataFrame({"label":   y_train}).to_csv(os.path.join(output_dir, "train_labels.csv"),   index=False)
pd.DataFrame({"context": X_test}).to_csv(os.path.join(output_dir,  "test_features.csv"),  index=False)
pd.DataFrame({"label":   y_test}).to_csv(os.path.join(output_dir,  "test_labels.csv"),    index=False)

print(f"Wrote train ({len(X_train)}) and test ({len(X_test)}) splits to {output_dir}")
print(f"Train 'none' share: {(pd.Series(list(y_train))=='none').mean()*100:.1f}%  "
      f"| Test 'none' share: {(y_test=='none').mean()*100:.1f}% (natural)")
print("Classes:", sorted(y.unique()))

Overwriting src/preprocess.py


## 4. Training script

This trains **one `sklearn.Pipeline`** (TF-IDF + classifier) and saves it as a single artefact. The classifier is chosen by the `--model-type` hyperparameter, so the **same script trains either of our two models**:
- `logreg` - the baseline Logistic Regression
- `svm` - Ria's Linear SVM (Model B)
- `tree` - the tree-ensemble model (RandomForest)

Metrics printed are **macro-F1** and **top-3 accuracy**; SageMaker captures them via the regex in the estimator's `metric_definitions`. MLflow logging is done by the notebook after the run (same as the lecturer's approach - the training container has no MLflow dependency).

In [6]:
%%writefile src/train.py
"""SageMaker Training Job - fits one sklearn Pipeline (TF-IDF + classifier).

The whole preprocessing+model is a single Pipeline, saved as model.joblib.
--model-type selects the classifier so the same script serves both models.
"""
import os
import argparse
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, accuracy_score, classification_report

parser = argparse.ArgumentParser()
parser.add_argument("--model-type", type=str, default="logreg")  # logreg | svm | tree
parser.add_argument("--random-state", type=int, default=42)
parser.add_argument("--team-id",   type=str, default=os.environ.get("TEAM_ID", "team16"))
parser.add_argument("--student-id",type=str, default=os.environ.get("STUDENT_ID", "s1602"))
parser.add_argument("--run-name",  type=str, default="sagemaker_pipeline_run")
parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
parser.add_argument("--test",  type=str, default=os.environ.get("SM_CHANNEL_TEST",  "/opt/ml/input/data/test"))
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

X_train = pd.read_csv(os.path.join(args.train, "train_features.csv"))["context"].fillna("")
y_train = pd.read_csv(os.path.join(args.train, "train_labels.csv"))["label"]
X_test  = pd.read_csv(os.path.join(args.test,  "test_features.csv"))["context"].fillna("")
y_test  = pd.read_csv(os.path.join(args.test,  "test_labels.csv"))["label"]
print(f"Train: {len(X_train)} | Test: {len(X_test)} | model-type: {args.model_type}")

# --- choose the classifier (this is how one pipeline serves both models) ---
def make_classifier(kind):
    if kind == "svm":
        # LinearSVC has no predict_proba; wrap for top-3 support.
        return CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", random_state=args.random_state))
    if kind == "tree":
        return RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=args.random_state, n_jobs=-1)
    # default: logreg baseline
    return LogisticRegression(max_iter=1000, class_weight="balanced",
                              random_state=args.random_state)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000, min_df=3, ngram_range=(1, 3),
        token_pattern=r"(?u)\b\w+\b|EMOJI_\w+")),
    ("clf", make_classifier(args.model_type)),
])
pipe.fit(X_train, y_train)

def top_k_acc(model, X, y, k=3):
    if not hasattr(model, "predict_proba"):
        return float("nan")
    proba = model.predict_proba(X)
    classes = model.classes_
    topk = classes[np.argsort(proba, axis=1)[:, -k:]]
    return float(np.mean([yt in row for yt, row in zip(y, topk)]))

y_pred = pipe.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
acc      = accuracy_score(y_test, y_pred)
top3     = top_k_acc(pipe, X_test, y_test, k=3)

# Print in a form SageMaker metric_definitions can capture.
print(f"test_macro_f1: {macro_f1:.4f}")
print(f"test_accuracy: {acc:.4f}")
print(f"test_top3: {top3:.4f}")
print("=== classification report ===")
print(classification_report(y_test, y_pred, zero_division=0))

# --- save the single Pipeline artefact ---
joblib.dump(pipe, os.path.join(args.model_dir, "model.joblib"))
print("Saved model.joblib (single Pipeline: TF-IDF + classifier)")

Overwriting src/train.py


## 5. Inference script

The endpoint handler. Because the model is one Pipeline, `model_fn` loads a single object and `predict_fn` runs raw text straight through it (TF-IDF happens inside). Input is `{"text": "..."}`; output is the predicted marker plus the top-3 suggestions - which is what the keyboard needs.

In [7]:
%%writefile src/inference.py
"""SageMaker inference handler - single Pipeline, text in / top-3 out."""
import os
import json
import joblib
import numpy as np

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(body, content_type="application/json"):
    if content_type != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")
    payload = json.loads(body)
    # accept {"text": "..."} or {"context": "..."} or a bare list of strings
    if isinstance(payload, dict):
        text = payload.get("text", payload.get("context", ""))
        return [text]
    if isinstance(payload, list):
        return [str(t) for t in payload]
    return [str(payload)]

def predict_fn(inputs, model):
    preds = model.predict(inputs)
    out = []
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(inputs)
        classes = model.classes_
        for i, p in enumerate(preds):
            order = np.argsort(proba[i])[::-1][:3]
            top3 = [{"marker": str(classes[j]), "score": round(float(proba[i][j]), 4)}
                    for j in order]
            out.append({"prediction": str(p), "top3": top3})
    else:
        for p in preds:
            out.append({"prediction": str(p), "top3": []})
    return out

def output_fn(prediction, accept="application/json"):
    return json.dumps(prediction), "application/json"

Overwriting src/inference.py


## 6. Confirm the corpus input location

The Processing step reads the raw corpus from S3. Notebook 01B staged it; this cell just confirms it is there.

In [8]:
s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_object(Bucket=BUCKET, Key=RAW_CORPUS_KEY)
    print("Corpus found:", f"s3://{BUCKET}/{RAW_CORPUS_KEY}")
except Exception as e:
    print("Corpus NOT found - run Notebook 01B first, or check the key.")
    print("Expected:", f"s3://{BUCKET}/{RAW_CORPUS_KEY}")
    raise

Corpus found: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/raw/smsCorpus_en_2015.03.09_all.json


## 7. Pipeline parameters

Following the lecturer's pattern, we expose the choices as pipeline parameters. The important one for us is `ModelType`, which selects the classifier - this is how the same pipeline trains either model. `MacroF1Gate` replaces the lecturer's AUC gate.

In [9]:
from sagemaker.workflow.parameters import ParameterString, ParameterFloat, ParameterInteger

p_model_type = ParameterString(name="ModelType", default_value="logreg")   # logreg | svm | tree
p_test_size  = ParameterFloat(name="TestSize", default_value=0.20)
p_min_count  = ParameterInteger(name="MinClassCount", default_value=100)
p_f1_gate    = ParameterFloat(name="MacroF1Gate", default_value=0.05)      # register if test macro-F1 >= this
print("Parameters defined.")

Parameters defined.


## 8. Step 1 - Processing (data preparation)

Runs `preprocess.py` as a SageMaker Processing job. Mirrors the lecturer's `ProcessingStep`, with our input being the corpus JSON and our output the train/test splits.

In [10]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"iti113-{TEAM_ID}-preprocess",
    sagemaker_session=pipeline_session,
)

step_process = ProcessingStep(
    name="PrepareSinglishData",
    processor=sklearn_processor,
    code="src/preprocess.py",
    inputs=[
        ProcessingInput(
            source=f"s3://{BUCKET}/{RAW_CORPUS_KEY}",
            destination="/opt/ml/processing/input",
        ),
        ProcessingInput(
            source="src/singlish_labelling.py",
            destination="/opt/ml/processing/input/module",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="train",
            source="/opt/ml/processing/output",
            destination=f"s3://{BUCKET}/{PREFIX}/processed"),
    ],
    job_arguments=[
        "--test-size", "0.20",
        "--min-class-count", "100",
    ],
)
print("Step 1 (ProcessingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 1 (ProcessingStep) defined.


## 9. Step 2 - Training

Runs `train.py` as a SageMaker Training job using the `SKLearn` estimator (framework 1.2-1, same as our ITI112 work). `metric_definitions` capture the printed macro-F1 / accuracy / top-3 so the pipeline can gate on them. `ModelType` is passed through so we can train either model.

In [11]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "model-type": p_model_type,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={"TEAM_ID": TEAM_ID, "STUDENT_ID": STUDENT_ID, "SEMESTER": SEMESTER},
    metric_definitions=[
        {"Name": "test_macro_f1", "Regex": "test_macro_f1: ([0-9\\.]+)"},
        {"Name": "test_accuracy", "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_top3",     "Regex": "test_top3: ([0-9\\.]+)"},
    ],
)

step_train = TrainingStep(
    name="TrainSinglishModel",
    estimator=estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"),
        "test": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"),
    },
)
print("Step 2 (TrainingStep) defined.")

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearn is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Step 2 (TrainingStep) defined.


### 9A. Model registry step

Following the lecturer's Notebook 03 pattern, we register the trained model into a **SageMaker Model Package Group** so each run's model is versioned. Registration is wired into the quality gate (Section 10): the model is registered **only if** it passes the macro-F1 threshold. This gives us model versioning (required by the rubric) and a clean "only good models get registered" flow.

> Student note: models are registered with `PendingManualApproval` status, so we consciously approve a version before it is used - a simple governance control.

In [12]:
from sagemaker.sklearn.model import SKLearnModel as PipelineSKLearnModel
from sagemaker.workflow.model_step import ModelStep

MODEL_PACKAGE_GROUP = f"iti113-{TEAM_ID}-singlish-models"

# Build a model object from the training step's output (for registration).
model_for_registry = PipelineSKLearnModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    entry_point="inference.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=pipeline_session,
)

step_register = ModelStep(
    name="RegisterSinglishModel",
    step_args=model_for_registry.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status="PendingManualApproval",
    ),
)
print("Model registry step defined. Group:", MODEL_PACKAGE_GROUP)

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


Model registry step defined. Group: iti113-team16-singlish-models


## 10. Step 3 - Quality gate and assemble the pipeline

The lecturer gates on test AUC-ROC. Since our problem is multi-class, we gate on **test macro-F1** instead: the pipeline proceeds only if the model clears a minimum bar. Then we assemble and upsert the pipeline, exactly as the lecturer does.

In [13]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.pipeline import Pipeline

condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_macro_f1"].Value,
    right=p_f1_gate,
)
step_gate = ConditionStep(
    name="MacroF1QualityGate",
    conditions=[condition],
    if_steps=[step_register],   # register the model ONLY if it passes the macro-F1 gate
    else_steps=[],
)
print("Step 3 (ConditionStep) defined.")

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[p_model_type, p_test_size, p_min_count, p_f1_gate],
    steps=[step_process, step_train, step_gate],   # step_register runs inside the gate
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print("No visual editor in this course - inspect in the SageMaker console under Pipelines, and CloudWatch.")

# 2. Start it through the pipeline (NOT the standalone job cells), SVM only.
execution = pipeline.start(parameters={"ModelType": "svm"})
print("Started:", execution.arn)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-

Step 3 (ConditionStep) defined.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "iti113-team16-singlish-pipeline" upserted.
No visual editor in this course - inspect in the SageMaker console under Pipelines, and CloudWatch.
Started: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team16-singlish-pipeline/execution/jjgzqban7z9d


## 11. Run the pipeline

Start an execution. To train the **baseline** leave `ModelType=logreg`. To train the other models, re-run with `ModelType=svm` or `ModelType=tree`. Each run trains one model through the identical pipeline - this is our evidence that the pipeline supports multiple models.

22-Aug-2026 - After champion Model identified as SVM, pipeline execution changed to SVM

In [14]:
sm = boto3.client("sagemaker", region_name=REGION)

# The pipeline was registered (upserted above). Set RUN_PIPELINE=True to launch a fresh
# gated execution (~10-15 min). A prior run already succeeded and populated the registry,
# so this defaults to False so a full Run-All does not launch duplicate executions.
RUN_PIPELINE = False

if RUN_PIPELINE:
    execution = pipeline.start(parameters={"ModelType": "svm"})
    print("Started:", execution.arn)
    execution.wait()
    print("Status:", execution.describe()["PipelineExecutionStatus"])
    for step in execution.list_steps():
        print(" -", step["StepName"], step["StepStatus"])
else:
    print("RUN_PIPELINE=False - not launching a new run. Recent executions:")
    for e in sm.list_pipeline_executions(PipelineName=PIPELINE_NAME,
            SortBy="CreationTime", SortOrder="Descending", MaxResults=3
            )["PipelineExecutionSummaries"]:
        print(" ", e["PipelineExecutionStatus"], e["PipelineExecutionArn"].split("/")[-1])

RUN_PIPELINE=False - not launching a new run. Recent executions:
  Executing jjgzqban7z9d
  Succeeded 5a90nudvfnpt
  Succeeded ssjoxhko15b4


## 11A. Train all three models through the same pipeline (multi-model evidence)

The requirement is that the MLOps pipeline supports **more than one model**. The `ModelType` parameter selects the classifier, so the *same pipeline* trains each model with no rebuild. This section runs all three (`logreg`, `svm`, `tree`) in turn and collects their metrics into one comparison table.

**Why this matters (student notes):**
- It is direct evidence that the pipeline serves multiple models (a rubric requirement).
- Because every model uses the identical preprocessing, split and metrics, the results are **fairly comparable** - this table is also a starting point for the model comparison in the report.
- `logreg` is the baseline. `svm` (Linear SVM, calibrated for top-3) and `tree` (Random Forest) are the models beyond it.

**Note on runtime:** each run is a full pipeline execution (a few minutes). The `svm` run may take a little longer because the calibration step does internal cross-validation. Run this section when you have ~15 minutes.

### 11A.1 Run each model through the pipeline

This loops over the three model types, starting a pipeline execution for each and waiting for it to finish. If a run fails, its status is printed so you can inspect that model's training job in CloudWatch.

In [15]:
# Optional: run all three model types through the SAME pipeline to evidence that the
# pipeline is model-agnostic. Expensive (~3 x 15 min) and the comparison is already
# captured for the report, so this defaults to False.
RUN_ALL_MODELS = False

model_types = ["logreg", "svm", "tree"]
execution_arns = {}
if RUN_ALL_MODELS:
    for mt in model_types:
        print(f"\n=== Starting pipeline for ModelType = {mt} ===")
        ex = pipeline.start(parameters={"ModelType": mt})
        execution_arns[mt] = ex.arn
        ex.wait()
        print(f"  {mt}: {ex.describe()['PipelineExecutionStatus']}")
else:
    print("RUN_ALL_MODELS=False - skipping multi-model runs (comparison already captured).")

RUN_ALL_MODELS=False - skipping multi-model runs (comparison already captured).


### 11A.2 Collect each model's metrics into a comparison table

For each model, we read the metrics from its training job. Because the jobs run in sequence, we match each model to its training job by creation order. The result is a clean comparison of the three models on the same evaluation.

In [16]:
import boto3, pandas as pd
sm = boto3.client("sagemaker", region_name=REGION)

def collect_model_metrics():
    """For recent completed TrainSinglishModel jobs, read the model-type from the job's
    hyperparameters and its metrics. Reading model-type directly (not guessing from order)
    means the labels are always correct even if runs are re-done or interleaved."""
    jobs = sm.list_training_jobs(StatusEquals="Completed", SortBy="CreationTime",
        SortOrder="Descending", MaxResults=20)
    seen = {}
    for j in jobs["TrainingJobSummaries"]:
        if "TrainSinglishModel" not in j["TrainingJobName"]:
            continue
        d = sm.describe_training_job(TrainingJobName=j["TrainingJobName"])
        hp = d.get("HyperParameters", {})
        # hyperparameter key is 'model-type' (may be quoted)
        mt = hp.get("model-type", hp.get("model_type", "unknown")).strip().strip('"')
        if mt in seen:      # keep only the most recent run per model type
            continue
        m = {x["MetricName"]: round(float(x["Value"]), 4)
             for x in d.get("FinalMetricDataList", [])}
        seen[mt] = {
            "Model": mt,
            "macro_F1": m.get("test_macro_f1"),
            "top3_acc": m.get("test_top3"),
            "accuracy": m.get("test_accuracy"),
            "training_job": j["TrainingJobName"],
        }
    return seen

results = collect_model_metrics()
order = [mt for mt in ["logreg", "svm", "tree"] if mt in results]
df_compare = pd.DataFrame([results[mt] for mt in order])[
    ["Model", "macro_F1", "top3_acc", "accuracy", "training_job"]]

print("Model comparison (each trained through the same pipeline):")
print(df_compare.to_string(index=False))
print("\nModel types found:", order)

Model comparison (each trained through the same pipeline):
Model  macro_F1  top3_acc  accuracy                                         training_job
  svm    0.1195    0.9637     0.873 pipelines-5a90nudvfnpt-TrainSinglishModel-3TOurseYXC

Model types found: ['svm']


### 11A.3 Notes on the comparison

> Fill in after running: which model has the best macro-F1, which the best top-3, and which you would pick as the champion. Remember top-3 is the product-relevant metric for a keyboard.
>
> This table is evidence for two things: (1) the pipeline supports multiple models, and (2) a fair, like-for-like comparison of the models. The modelling write-up (error analysis, champion justification) is developed further in the model-development notebook.

## 12. Log the run to MLflow (after the pipeline)

Same approach as the lecturer and as our Notebook 01A: the training container stays free of MLflow, and we log to our Team 16 MLflow app **after** a successful run. This records parameters and metrics so runs are reproducible.

This cell reuses the exact MLflow pattern that worked in our progress-check 01A (tracking URI = the app ARN, our experiment, tags + params + metrics). The difference here is we log the **real** pipeline metrics (macro-F1, top-3), not placeholder values, and we pull them from the training job.

In [17]:
import mlflow, boto3

# ---- Team 16 MLflow app (same as Notebook 01A) ----
# Option 1: paste your app ARN from 01A (this is what 01A used as the tracking URI).
MLFLOW_APP_ARN = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-5VMLPBFVZIYL"
EXPERIMENT_NAME = "ITI113/team16/Experiment1"
ARTIFACT_STORE_URI = f"s3://{BUCKET}/{PREFIX.rsplit('/data',1)[0]}/mlflow-app-artifacts/"

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(EXPERIMENT_NAME)

# ---- pull the real metrics from the just-finished training job ----
# The TrainingStep captured test_macro_f1 / test_accuracy / test_top3 via metric_definitions.
sm = boto3.client("sagemaker", region_name=REGION)
model_type_run = "svm"   # match what you ran in Section 11

def get_pipeline_training_metrics():
    """Find the most recent COMPLETED training job from our pipeline and read its metrics.
    Pipeline-created training jobs are named like 'pipelines-XXXX-TrainSinglishModel-XXXX',
    so we match on the step name rather than a fixed prefix."""
    jobs = sm.list_training_jobs(
        StatusEquals="Completed", SortBy="CreationTime",
        SortOrder="Descending", MaxResults=10)
    name = None
    for j in jobs["TrainingJobSummaries"]:
        if "TrainSinglishModel" in j["TrainingJobName"]:
            name = j["TrainingJobName"]
            break
    if name is None:
        print("No TrainSinglishModel training job found.")
        return {}
    print("Reading metrics from:", name)
    desc = sm.describe_training_job(TrainingJobName=name)
    out = {}
    for m in desc.get("FinalMetricDataList", []):
        out[m["MetricName"]] = float(m["Value"])
    return out

metrics = get_pipeline_training_metrics()
print("Metrics to log:", metrics)
#team16_s1601_model_b_linear_svm

run_name = f"{TEAM_ID}_{STUDENT_ID}_model_{model_type_run}"
with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": "ITI113", "semester": SEMESTER,
        "team_id": TEAM_ID, "student_id": STUDENT_ID,
        "tracking_backend": "sagemaker_mlflow_app",
        "purpose": "pipeline_run", "project_name": PROJECT,
        "model_type": model_type_run,
    })
    mlflow.log_params({
        "team_id": TEAM_ID, "student_id": STUDENT_ID, "region": REGION,
        "pipeline": PIPELINE_NAME, "model_type": model_type_run,
        "artifact_store_uri": ARTIFACT_STORE_URI, "project_name": PROJECT,
    })
    if metrics:
        mlflow.log_metrics(metrics)
    print("Logged run:", run.info.run_id)

print("Done. Check the MLflow UI (presigned URL from 01A) - the run appears under", EXPERIMENT_NAME)

Reading metrics from: pipelines-5a90nudvfnpt-TrainSinglishModel-3TOurseYXC
Metrics to log: {'test_top3': 0.963699996471405, 'test_accuracy': 0.8730000257492065, 'test_macro_f1': 0.11949999630451202}
Logged run: 19a74b40ac174f8ab699ae04ed956361
🏃 View run team16_s1602_model_svm at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/19a74b40ac174f8ab699ae04ed956361
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Done. Check the MLflow UI (presigned URL from 01A) - the run appears under ITI113/team16/Experiment1


## 13. Summary and next steps

**What this notebook does:**
- Prepares the corpus (Option C labelling via the shared module) as a Processing step.
- Trains a single TF-IDF + classifier Pipeline as a Training step, gated on macro-F1.
- Uses one `ModelType` parameter so the **same pipeline trains and serves both models** (SVM and tree ensemble), meeting the two-model requirement.
- Logs runs to the Team 16 MLflow app for reproducibility.

**Next (student notes):**
- Run the pipeline once per model (`logreg`, `svm`, `tree`) and capture the metrics and console/CloudWatch screenshots as evidence.
- Deploy the chosen model to a serverless endpoint (this can be added as a ModelStep, following the lecturer's ModelStep pattern).
- Build the Gradio demo (Notebook 04) to call the endpoint.
- Inspect any pipeline errors using the lecturer's troubleshooting notebook (no visual editor in this setup).

---

## 14. Deploy the model to a serverless endpoint

This section deploys the model that the pipeline trained, so a front-end (the Gradio demo in Notebook 04) can call it. The deployment mechanics follow the same single-Pipeline pattern we used in the ITI112 assignment: one artefact (`model.joblib`), one `inference.py` handler, deployed to a SageMaker **serverless** endpoint. Everything else - the endpoint name, the JSON input/output format, and the security model - follows this project and the lecturer's Notebook 04 conventions.

**Assumptions / decisions (student notes):**
- We deploy the artefact the **pipeline** produced (the `TrainSinglishModel` job output in S3), not a separately trained model - this demonstrates the pipeline → endpoint flow.
- The endpoint is **serverless** (no server to manage; suitable for a demo), per the lecturer's materials.
- Endpoint name follows the lecturer's `iti113-{team}-{project}` pattern: `iti113-team16-singlish-keyboard`.
- Input is `{"text": "..."}`; output is the predicted marker plus top-3 (our `inference.py` already returns this). This matches how Notebook 04 invokes an endpoint with `boto3` and JSON.
- Security (per the lecturer's note): the endpoint is invoked using the Studio execution role. We only use our own team endpoint and do not share the demo link widely.

**Run order after a kernel restart:** run cell 0 (install, if needed) and cell 1 (config) to rebuild the session variables, then run this section. You do **not** need to re-run the pipeline (cells 2–11) - the trained model already exists in S3.

### 14.1 Resolve the champion model from the registry

Deterministic: deploy the APPROVED registry version (v8), not the newest training job.

In [18]:
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
MODEL_PKG_GROUP = "iti113-team16-singlish-models"

# Deploy the APPROVED champion from the model registry - deterministic, NOT the newest
# training job. Ascending + MaxResults=1 returns the FIRST approved version (v8), the
# champion that is (and stays) deployed. This avoids accidentally picking a later run.
approved = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PKG_GROUP, ModelApprovalStatus="Approved",
    SortBy="CreationTime", SortOrder="Ascending", MaxResults=1
)["ModelPackageSummaryList"]
if not approved:
    raise RuntimeError("No Approved model package found in the registry.")
champion_arn = approved[0]["ModelPackageArn"]
desc = sm.describe_model_package(ModelPackageName=champion_arn)
MODEL_DATA_URI = desc["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
print("Champion registry version:", champion_arn.split("/")[-1])
print("Model artefact           :", MODEL_DATA_URI)
print("Is champion (3QzRtEmYjC) :", "3QzRtEmYjC" in MODEL_DATA_URI)

Champion registry version: 8
Model artefact           : s3://sagemaker-ap-southeast-1-044528205969/pipelines-s9t0j8thg6dz-TrainSinglishModel-3QzRtEmYjC/output/model.tar.gz
Is champion (3QzRtEmYjC) : True


### 14.2 Create the SageMaker model (Pipeline artefact + inference.py)

We wrap the artefact in an `SKLearnModel` with our `inference.py` as the entry point, using the same framework version as training (1.2-1). This is the ITI112 deployment pattern: one model object, one handler, no separate preprocessing container.

In [19]:
from sagemaker.sklearn.model import SKLearnModel

ENDPOINT_NAME = f"iti113-{TEAM_ID}-singlish-keyboard"
MODEL_NAME    = f"iti113-{TEAM_ID}-singlish-model"

# SAFETY: the endpoint already serves the approved champion (v8). A full Run-All must NOT
# clobber it. Set DEPLOY=True only when you deliberately intend to (re)deploy.
DEPLOY = False

if DEPLOY:
    sklearn_model = SKLearnModel(
        model_data=MODEL_DATA_URI, role=role, entry_point="inference.py",
        source_dir=LOCAL_PIPELINE_SRC, framework_version="1.2-1", py_version="py3",
        name=MODEL_NAME, sagemaker_session=sm_session,
    )
    print("SageMaker model defined:", MODEL_NAME)
else:
    print("DEPLOY=False - not (re)creating the model. Endpoint keeps serving the champion.")

DEPLOY=False - not (re)creating the model. Endpoint keeps serving the champion.


### 14.3 Deploy to a serverless endpoint

Serverless config: memory and max concurrency sized small - our model is light (TF-IDF + classifier). Deployment takes a few minutes.

In [20]:
from sagemaker.serverless import ServerlessInferenceConfig

if DEPLOY:
    serverless_config = ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=5)
    predictor = sklearn_model.deploy(
        serverless_inference_config=serverless_config, endpoint_name=ENDPOINT_NAME)
    print("Deployed endpoint:", ENDPOINT_NAME)
else:
    print("DEPLOY=False - skipping deploy. Existing endpoint left untouched.")

DEPLOY=False - skipping deploy. Existing endpoint left untouched.


### 14.4 Test the endpoint with boto3

We invoke the endpoint the same way Notebook 04's Gradio UI will: a JSON `{"text": "..."}` request via `sagemaker-runtime`. The response is the predicted marker and the top-3 suggestions.

In [21]:
import json
rt = boto3.client("sagemaker-runtime", region_name=REGION)

def predict_marker(text):
    resp = rt.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"text": text}),
    )
    return json.loads(resp["Body"].read())

for msg in ["ok can", "so happy", "dunno why like that", "meeting at 3pm"]:
    out = predict_marker(msg)
    print(f'{msg!r:35s} -> {out}')

'ok can'                            -> [{'prediction': 'none', 'top3': [{'marker': 'none', 'score': 0.5315}, {'marker': 'emo_joy', 'score': 0.3027}, {'marker': 'lah', 'score': 0.036}]}]
'so happy'                          -> [{'prediction': 'none', 'top3': [{'marker': 'none', 'score': 0.5189}, {'marker': 'emo_joy', 'score': 0.3198}, {'marker': 'emo_playful', 'score': 0.0482}]}]
'dunno why like that'               -> [{'prediction': 'none', 'top3': [{'marker': 'none', 'score': 0.5657}, {'marker': 'emo_joy', 'score': 0.125}, {'marker': 'emo_sad', 'score': 0.0652}]}]
'meeting at 3pm'                    -> [{'prediction': 'none', 'top3': [{'marker': 'none', 'score': 0.5529}, {'marker': 'eh', 'score': 0.1641}, {'marker': 'emo_joy', 'score': 0.084}]}]


### 14.5 Confirm the endpoint is InService

A quick status check - the Gradio demo (Notebook 04) needs this to be `InService`.

In [22]:
desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
cfg  = sm.describe_endpoint_config(EndpointConfigName=desc["EndpointConfigName"])
md   = sm.describe_model(ModelName=cfg["ProductionVariants"][0]["ModelName"])
print("Endpoint :", ENDPOINT_NAME)
print("Status   :", desc["EndpointStatus"])
print("Artifact :", md["PrimaryContainer"]["ModelDataUrl"])
print("Is champion SVM (3QzRtEmYjC):", "3QzRtEmYjC" in md["PrimaryContainer"]["ModelDataUrl"])
print("\nWhen InService + champion=True, the endpoint is ready for the Gradio demo (Notebook 04).")

Endpoint : iti113-team16-singlish-keyboard
Status   : InService
Artifact : s3://sagemaker-ap-southeast-1-044528205969/pipelines-s9t0j8thg6dz-TrainSinglishModel-3QzRtEmYjC/output/model.tar.gz
Is champion SVM (3QzRtEmYjC): True

When InService + champion=True, the endpoint is ready for the Gradio demo (Notebook 04).


### 14.6 (Optional) Clean up

Serverless endpoints only bill per request, so they are cheap to leave up for the demo. Delete when done to keep the account tidy. **Leave the endpoint up until after the presentation.**

In [23]:
# Uncomment to delete the endpoint when completely finished (e.g. after the 26th):
# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# print("Deleted endpoint:", ENDPOINT_NAME)
print("Endpoint left running for the demo. Delete after the presentation if desired.")

Endpoint left running for the demo. Delete after the presentation if desired.


## 15. Registry & endpoint verification (evidence)

Read-only checks that produce clean evidence on a Run-All: the model registry versions/approvals, and confirmation the live endpoint serves the champion.

In [24]:
# Registry evidence: versions and approval status (v8 = deployed champion, Approved)
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
for p in sm.list_model_packages(
        ModelPackageGroupName="iti113-team16-singlish-models",
        SortBy="CreationTime", SortOrder="Descending", MaxResults=8
        )["ModelPackageSummaryList"]:
    print(p["ModelPackageArn"].split("/")[-1], p["ModelApprovalStatus"], p["CreationTime"])

11 Approved 2026-08-23 05:20:20.873000+00:00
10 PendingManualApproval 2026-08-23 04:55:53.766000+00:00
9 PendingManualApproval 2026-08-23 04:55:18.035000+00:00
8 Approved 2026-08-22 08:29:38.740000+00:00
7 PendingManualApproval 2026-08-22 08:29:06.123000+00:00
6 PendingManualApproval 2026-08-22 08:21:58.845000+00:00
5 PendingManualApproval 2026-08-22 08:13:31.843000+00:00
4 PendingManualApproval 2026-08-21 10:09:47.829000+00:00
